In [37]:
import numpy as np
from scipy.stats import multivariate_normal

class GaussianMixtureModel:
    def __init__(self, M, n_features, lamda):
        self.lamda = lamda
        self.M = M
        self.n_features = n_features
        self.means = np.zeros((M, n_features))
        self.covariances = [np.eye(n_features) for _ in range(M)]
        self.weights = np.ones(M) / M
        self.epsilon = 1e-6  # 用于数值稳定性的小常数
        self.prev_log_likelihood = -np.inf  # 初始化 prev_log_likelihood 属性

    def log_likelihood(self, X, responsibilities):
        weights_2d = self.weights[:, np.newaxis]
        gaussian_pdfs = [multivariate_normal.pdf(X, mean=m, cov=c + self.epsilon * np.eye(self.n_features)) for m, c in zip(self.means, self.covariances)]
        sum_pdf = np.sum(weights_2d * np.array(gaussian_pdfs), axis=0)
        return np.sum(responsibilities * np.log(sum_pdf + self.epsilon))

    def penalized_log_likelihood(self, X):
        log_likelihood = self.log_likelihood(X, self.E_step(X))
        Df = self.n_features * (self.n_features + 1) / 2 + self.n_features + 1  # 自由参数数量
        penalty = - X.shape[0] * self.lamda * Df * np.sum(np.log(self.weights + self.epsilon) - np.log(self.epsilon))
        return log_likelihood + penalty

    def E_step(self, X):
        responsibilities = np.zeros((self.M, X.shape[0]))
        for i in range(self.M):
            weight_density = self.weights[i] * multivariate_normal.pdf(X, mean=self.means[i], cov=self.covariances[i] + self.epsilon * np.eye(self.n_features))
            responsibilities[i] = weight_density
        responsibilities /= np.sum(responsibilities, axis=0)
        return responsibilities

    def modified_E_step(self, X):
        responsibilities = self.E_step(X)
        return responsibilities

    def modified_M_step(self, X, responsibilities):
        Df = self.n_features * (self.n_features + 1) / 2 + self.n_features + 1  # 自由参数数量
        for i in range(self.M):
            N_i = np.sum(responsibilities[i] - self.lamda * Df)
            self.weights[i] = max(0, N_i / (X.shape[0] * (1 - self.M * self.lamda * Df)))
            if self.weights[i] > 1e-4:
                self.means[i] = np.dot(responsibilities[i], X) / np.sum(responsibilities[i])
                diff = X - self.means[i]
                self.covariances[i] = np.dot(responsibilities[i] * diff.T, diff) / np.sum(responsibilities[i])
                self.covariances[i] += self.epsilon * np.eye(self.n_features)  # 添加正则化项

    def fit_modified(self, X, max_iter=100):
        for _ in range(max_iter):
            responsibilities = self.modified_E_step(X)
            self.modified_M_step(X, responsibilities)
            current_log_likelihood = self.penalized_log_likelihood(X)
            if np.abs(current_log_likelihood - self.prev_log_likelihood) < 1e-4:
                break
            self.prev_log_likelihood = current_log_likelihood

        # 删除权重接近零的成分
        significant_components = self.weights > 1e-4
        self.weights = self.weights[significant_components]
        self.means = self.means[significant_components]
        self.covariances = [self.covariances[i] for i in range(len(significant_components)) if significant_components[i]]
        self.M = len(self.weights)

        return self.weights, self.means, self.covariances

# 设置随机种子以获得可重现的结果
np.random.seed(42)
# 生成数据集
n = 600  # 样本数量
_means = np.array([[-1, 1], [1, 1], [0, -np.sqrt(2)]])
_covariances = [np.diag([2, 0.2 ** 2])] * 3
_mixing_probabilities = [1 / 3, 1 / 3, 1 / 3]
X = np.concatenate([np.random.multivariate_normal(means, covariances, int(n * mixing_probabilities))
                    for mixing_probabilities, means, covariances in zip(_mixing_probabilities, _means, _covariances)]) 

# 定义一个函数来计算BIC值
def calculate_bic(gmm, X):
    global bic
    Df = gmm.n_features * (gmm.n_features + 1) / 2 + gmm.n_features + 1
    log_likelihood = gmm.log_likelihood(X, gmm.E_step(X))
    for m in range(gmm.M):
        bic=sum(np.log(gmm.weights[m] * multivariate_normal.pdf(X, mean=gmm.means[m], cov=gmm.covariances[m] + gmm.epsilon * np.eye(gmm.n_features)))) - 1/2 * gmm.M * Df * np.log(len(X))
    return bic

# 寻找最佳lamda和对应的BIC值
best_bic = float('inf')
best_lamda = 0
best_gmm = None

# 设置lamda的尝试范围和尝试次数
lamda_range = np.linspace(0.1, 0.215, 20)
for lamda in lamda_range:
    # 对每个lamda值运行300次
    bic_values = []
    for _ in range(300):
        # 创建GMM实例，设置当前的lamda值
        gmm = GaussianMixtureModel(M=10, n_features=2, lamda=lamda)
        # 拟合模型
        gmm.fit_modified(X)
         # 计算BIC值
        bic = calculate_bic(gmm, X)
        bic_values.append(bic)
    
    # 找出使得BIC最大的lamda
    best_idx = np.argmax(bic_values)
    if bic_values[best_idx] < best_bic:
        best_lamda = lamda
        # 假设我们存储了每个lamda值对应的最佳GMM实例
        best_gmm = GaussianMixtureModel(M=10, n_features=2, lamda=lamda)

# 输出最佳lamda
print(f"Best lamda: {best_lamda}")

# 使用best_lamda作为初始值重新运行高斯混合模型类得到最佳成分数best_M
best_gmm = GaussianMixtureModel(M=10, n_features=2, lamda=best_lamda)
best_gmm.fit_modified(X)

# 输出最佳成分数和对应的参数
print(f"Best number of components: {best_gmm.M}")
print("Weights:", best_gmm.weights)
print("Means:", best_gmm.means)
print("Covariances:", best_gmm.covariances)



Best lamda: 0.215
Best number of components: 10
Weights: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
Means: [[0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]
 [0.068373   0.20094563]]
Covariances: [array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.18550144,  1.32496518]]), array([[ 2.47218498, -0.18550144],
       [-0.1855014